# Unit 2 exercises: popularity, quality, and bias

Run the recsys data generators before you begin. These exercises build the first real retrieval path -- popularity -- score it honestly against Unit 1's random floor, and then measure the bias the scoreboard quietly rewards. Every count is computed on the `train` split only; `val` is the sealed holdout we grade against and `test` stays sealed tighter still. Each answer is left for you to complete; no solutions live here.

In [ ]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd

data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions = pd.read_csv(data_dir / "interactions.csv.gz")
interactions_path = data_dir / "interactions.csv.gz"
rows = interactions.to_dict("records")

## Exercise 1 - Count popularity on the training split only

Popularity is a count: how many readers gave each book a positive interaction (`label == 1`) on the `train` split. Filter `interactions` to positive train rows, group by `item_id`, and count, then sort the counts from most-read to least-read. Record the most-read book's `item_id` and its count. Finally, confirm that your filter never touched `val` or `test` by checking that the splits remaining after filtering are exactly `{"train"}`.

In [ ]:
train = interactions[interactions["split"] == "train"]
train_positives = None  # TODO: keep only label == 1 rows of `train`
counts = None           # TODO: groupby item_id, size, sort descending
top_item_id = None      # TODO: item_id of the most-read book
top_count = None         # TODO: its positive count
splits_used = None       # TODO: the set of splits present in your filtered frame (expect {"train"})

## Exercise 2 - Fit the PopularityRetrievalPath and retrieve without repeats

The package ships this exact count as `bookrec.PopularityRetrievalPath`. Its `fit` takes an iterable of row mappings with keys `reader_id, item_id, split, label` (the `rows` built in setup) and a `catalog=` of item ids; leakage safety lives inside the code. Fit the path, then confirm its `.counts` dictionary matches your by-hand counts from Exercise 1. Retrieve the top 5 items for an arbitrary reader, passing `context={"seen": ...}` with two already-read item ids, and check that neither seen id appears in the result and that the path is reader-independent (two different queries give the same list).

In [ ]:
popularity = None        # TODO: bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
counts_match = None      # TODO: compare popularity.counts against your Exercise-1 counts (bool)
seen = {catalog_ids[0], catalog_ids[1]}
top5 = None              # TODO: popularity.retrieve(query=..., context={"seen": seen}, k=5)
top5_item_ids = []       # TODO: the item ids of top5
no_seen_leaked = None    # TODO: bool - none of `seen` appears in top5_item_ids
reader_independent = None  # TODO: bool - a second query with a different reader gives the same ids

## Exercise 3 - Register the path and beat the random floor

Load `cold_partitions.json` and build the set of cold readers (they must never be scored). Register your fitted `popularity` path in a fresh `bookrec.PathRegistry` (a name/artifact collision raises). Then score both `popularity` and a `bookrec.RandomRetrievalPath(catalog_ids, seed=0)` floor on the `val` scoreboard with `bookrec.run_validation_scoreboard(..., k=10, cold_readers=cold_readers)`. Record each path's `hit_rate_at_k`, the number of readers scored, and the ratio popularity/random. In one sentence, say what "beats the floor" means here and why popularity can win it before we know anything about a reader.

In [ ]:
cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

registry = None          # TODO: bookrec.PathRegistry(); register `popularity` in it
random_floor = bookrec.RandomRetrievalPath(catalog_ids, seed=0)

pop_board = None         # TODO: run_validation_scoreboard(popularity, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)
random_board = None      # TODO: the same scoreboard for random_floor
readers_scored = None    # TODO: pop_board.readers
ratio = None             # TODO: pop_board.hit_rate_at_k / random_board.hit_rate_at_k
beats_floor_explanation = "TODO"  # one sentence

## Exercise 4 - Apply the weighted rating and predict the re-ranking

The weighted (Bayesian-shrinkage) rating is $\text{score} = \frac{v\,R + m\,C}{v + m}$, where $R$ is an item's own positive rate, $v$ its exposure, $C$ the global positive rate, and $m$ the prior strength in pseudo-counts. For the four given items below (positives and exposures), use `bookrec.weighted_rating(positives, exposures, m=..., global_rate=C)` to compute scores at `m = 0` (which recovers the raw rate $R$) and at `m = 10`. Then, WITHOUT recomputing, predict in a short list how the top-to-bottom ordering changes between the raw-rate ranking and the `m = 10` ranking. Two items are thin-but-perfect (both rate $1.0$ on tiny evidence): focus on **the 3-of-3 item (index 0)** and report how far it is demoted -- which rank it holds on raw rate and which rank it falls to at `m = 10`.

In [ ]:
positives = np.array([3, 40, 9000, 1])
exposures = np.array([3, 50, 10000, 1])
C = 0.5  # given global positive rate for this exercise

raw_scores = None        # TODO: bookrec.weighted_rating(positives, exposures, m=0.0, global_rate=C)
shrunk_scores = None     # TODO: bookrec.weighted_rating(positives, exposures, m=10.0, global_rate=C)
raw_order = []           # TODO: item indices sorted best-first by raw_scores
shrunk_order = []        # TODO: item indices sorted best-first by shrunk_scores
demoted_thin_item = None  # TODO: the 3-of-3 item, index 0 - report how far shrinkage pulls it down
ranking_change = "TODO"  # one or two sentences

## Exercise 5 - Score the quality ranking and explain the biased metric

Quality looks like the better idea, so hold it to the same scoreboard. Compute a weighted-rating score for every catalog item (on train exposures/positives, `m = 10`, global rate `C`), wrap it in a tiny reader-independent path that ranks by that score (structurally the popularity path's twin -- build `bookrec.Candidate` objects and return `bookrec.rank(candidates, n=k)`, excluding `context["seen"]`), and score it on `val` with the same cold-reader exclusion. You should find it lands far below the count path. Explain in two or three sentences WHY -- what about how the `val` holdout was built makes a quality-first ranker lose, and why that is "popularity bias living in the metric".

In [ ]:
# exposures/positives per catalog item, on TRAIN only (aligned to catalog_ids)
train = interactions[interactions["split"] == "train"]
train_positives = train[train["label"] == 1]
exposure_by = train.groupby("item_id").size()
positive_by = train_positives.groupby("item_id").size()
exposure_map = {int(i): int(v) for i, v in zip(exposure_by.index, exposure_by.values)}
positive_map = {int(i): int(v) for i, v in zip(positive_by.index, positive_by.values)}
item_exposures = np.array([exposure_map.get(i, 0) for i in catalog_ids])
item_positives = np.array([positive_map.get(i, 0) for i in catalog_ids])
C_global = float(item_positives.sum() / item_exposures.sum())

quality_scores = None    # TODO: bookrec.weighted_rating(item_positives, item_exposures, m=10.0, global_rate=C_global)

class QualityRetrievalPath:
    name = "quality"
    def __init__(self, item_ids, scores):
        self._pairs = None  # TODO: list of (int item_id, float score) pairs
    def retrieve(self, query, context, k):
        seen = set(context["seen"]) if "seen" in context else set()
        candidates = None  # TODO: bookrec.Candidate(item_id, score, "quality") for unseen pairs
        return None        # TODO: bookrec.rank(candidates, n=k)

quality_board = None     # TODO: run_validation_scoreboard(QualityRetrievalPath(...), interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)
quality_hit = None       # TODO: quality_board.hit_rate_at_k
why_below_popularity = "TODO"   # two or three sentences

## Exercise 6 - Measure the bias: catalog coverage and head-share

A recommender that wins accuracy by pandering to the head serves almost none of the catalog. Build the head set with `bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)` (the top 10% of items by train positive count). Collect each scored reader's top-k list for the `popularity` path and for the `random_floor` path over the SAME scored-reader set the scoreboard used (non-cold readers with at least one unseen positive `val` item), then compute `bookrec.catalog_coverage(recs, catalog_ids)` and `bookrec.head_share(recs, head)` for each. Record the four numbers and say in one or two sentences which path is fairer to the catalog and what bias the gap reveals.

In [ ]:
head = None              # TODO: bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)

def scored_recommendations(path):
    """Reproduce the scoreboard's scored-reader set and collect each one's top-k list."""
    catalog_set = set(catalog_ids)
    seen_by = defaultdict(set)
    val_by = defaultdict(set)
    for row in rows:
        if row["item_id"] not in catalog_set or row["label"] != 1:
            continue
        if row["split"] == "train":
            seen_by[row["reader_id"]].add(row["item_id"])
        elif row["split"] == "val":
            val_by[row["reader_id"]].add(row["item_id"])
    recommendations = []
    for reader_id in sorted(val_by):
        seen = seen_by[reader_id]
        if reader_id in cold_readers or not (val_by[reader_id] - seen):
            continue
        recs = None  # TODO: path.retrieve(reader_id, {"seen": seen}, 10)
        recommendations.append(recs)
    return recommendations

pop_recs = None          # TODO: scored_recommendations(popularity)
random_recs = None       # TODO: scored_recommendations(random_floor)
pop_coverage = None      # TODO: bookrec.catalog_coverage(pop_recs, catalog_ids)
pop_head_share = None    # TODO: bookrec.head_share(pop_recs, head)
random_coverage = None   # TODO: bookrec.catalog_coverage(random_recs, catalog_ids)
random_head_share = None  # TODO: bookrec.head_share(random_recs, head)
fairer_path_explanation = "TODO"  # one or two sentences

## Exercise 7 - Challenge: shrinkage as a convex combination

Show algebraically that the weighted rating is a convex combination of the item's own rate $R$ and the global rate $C$: rewrite $\frac{v R + m C}{v + m}$ as $w R + (1 - w) C$ and give $w$ in terms of $v$ and $m$, verifying $0 \le w \le 1$. Confirm the two limits ($m \to 0 \Rightarrow$ score $\to R$; $m \to \infty \Rightarrow$ score $\to C$) and show that as $m$ grows every score contracts toward $C$. Then predict: as $m \to \infty$ the ranking approaches ordering by $v\,(R - C)$ -- argue whether the **head-share** of the resulting whole-catalog quality ranker should rise or fall as $m$ grows, and why (think about what $v\,(R - C)$ favours: high-exposure items, which are the head). Verify the convex-combination identity numerically on the given demo arrays for one value of $m$ (equality to floating-point tolerance). Then, using the whole-catalog weighted scores from Exercise 5, `bookrec.head_ids_from_counts`, and the `scored_recommendations` helper from Exercise 6 (seed $=0$), measure the head-share of the quality ranker at a small $m$ (raw rate, $m \to 0$) and at a large $m$ and confirm it moves in the direction you predicted.

In [ ]:
v = np.array([3.0, 50.0, 10000.0, 1.0])
R = np.array([1.0, 0.8, 0.9, 1.0])
C_demo = 0.5
m = 10.0

w = None                 # TODO: the convex weight on R, in terms of v and m
score_direct = None      # TODO: bookrec.weighted_rating((v * R).astype(int), v.astype(int), m=m, global_rate=C_demo)
score_convex = None      # TODO: w * R + (1 - w) * C_demo
identity_holds = None    # TODO: bool - np.allclose(score_direct, score_convex)
w_in_unit_interval = None  # TODO: bool - every w in [0, 1]
head_share_direction = "TODO"  # 'rise' or 'fall' as m grows, with a one-sentence reason

# measure it: head-share of the whole-catalog quality ranker (Exercise 5 arrays) at small vs large m
head = None              # TODO: bookrec.head_ids_from_counts(popularity.counts, catalog_ids, fraction=0.10)
raw_quality = None       # TODO: bookrec.weighted_rating(item_positives, item_exposures, m=0.0, global_rate=C_global)
big_quality = None       # TODO: bookrec.weighted_rating(item_positives, item_exposures, m=1000.0, global_rate=C_global)
raw_head_share = None    # TODO: bookrec.head_share(scored_recommendations(QualityRetrievalPath(catalog_ids, raw_quality)), head)
big_head_share = None    # TODO: head-share of the large-m quality ranker, the same way

## Exercise 8 - Challenge: find the m that makes two items tie

Two items have exposures and rates $(v_1, R_1)$ and $(v_2, R_2)$ with the global rate $C$. Solve $\frac{v_1 R_1 + m C}{v_1 + m} = \frac{v_2 R_2 + m C}{v_2 + m}$ for the prior strength $m$ that makes their weighted ratings exactly equal. Cross-multiply, cancel the $m^2 C$ terms, and isolate $m$; report the closed form and the condition under which a finite positive solution exists (and what it means when it does not). Use the given numbers to compute $m^\star$, then verify by evaluating `bookrec.weighted_rating` at that $m$ that the two scores agree to floating-point tolerance.

In [ ]:
v1, R1 = 3, 1.0
v2, R2 = 10000, 0.9
C = 0.5

m_star = None            # TODO: closed-form m that equalises the two weighted ratings
exists_condition = "TODO"  # when is a finite positive m^star possible?
# verification
pos = np.array([int(round(v1 * R1)), int(round(v2 * R2))])
exp = np.array([v1, v2])
scores_at_m_star = None  # TODO: bookrec.weighted_rating(pos, exp, m=m_star, global_rate=C)
scores_agree = None      # TODO: bool - the two entries of scores_at_m_star match to tolerance